## Section 1 -- Background and Hyperparameter Rationale

This notebook tunes the hyperparameters of a **Temporal Convolutional Network (TCN)**
for binary EEG seizure detection (ictal vs non-ictal) using **Optuna** with the
Tree-structured Parzen Estimator (TPE) sampler.

### Receptive field formula

A TCN with `L` layers, kernel size `k`, and exponential dilation schedule `d_l = 2^l`
has a receptive field (RF) of:

```
RF = 2^L * (k - 1)
```

At 500 Hz, 1 second = 500 samples. We enforce RF >= 500 as a hard constraint.

**Worked examples:**

| L (layers) | k (kernel) | RF (samples) | RF (seconds) | Passes check? |
|------------|------------|--------------|--------------|---------------|
| 5          | 3          | 64           | 0.13         | No            |
| 6          | 7          | 384          | 0.77         | No            |
| 7          | 5          | 512          | 1.02         | Yes           |
| 8          | 3          | 512          | 1.02         | Yes           |
| 9          | 7          | 3072         | 6.14         | Yes           |

### Hyperparameters to tune

| Hyperparameter | Role                                      | Type         | Range / Choices           |
|----------------|-------------------------------------------|--------------|---------------------------|
| num_layers     | Expands RF exponentially; controls depth   | int          | 5 -- 9                    |
| kernel_size    | Expands RF linearly; odd values only       | categorical  | 3, 5, 7                   |
| num_filters    | Representational width per layer           | categorical  | 32, 64, 128               |
| dropout        | Spatial dropout rate per conv block        | float        | 0.10 -- 0.50 (step 0.05)  |
| learning_rate  | AdamW step size                            | float (log)  | 1e-4 -- 1e-2              |
| weight_decay   | L2 regularisation on weights               | float (log)  | 1e-5 -- 1e-3              |
| batch_size     | Segments per gradient update               | categorical  | 16, 32, 64                |

**Justification:**

- **num_layers 5--9:** minimum of 5 ensures RF >= 500 for kernel_size >= 3;
  beyond 9 layers the RF exceeds the segment length and offers no benefit.
- **kernel_size 3/5/7:** odd values only -- even kernels cause asymmetric
  causal padding and introduce subtle temporal boundary artefacts.
- **num_filters 32/64/128:** powers of two for efficient GPU memory alignment;
  values above 128 risk overfitting given the limited number of mice.
- **dropout 0.10--0.50:** lower bound prevents excessive regularisation that
  causes underfitting; upper bound prevents training collapse.
- **learning_rate 1e-4--1e-2 on log scale:** log scale is essential because
  the optimal lr can differ by orders of magnitude across configurations.
- **weight_decay 1e-5--1e-3 on log scale:** complements dropout; searched
  on log scale for the same reason as learning_rate.
- **batch_size 16/32/64:** values below 16 produce noisy gradient estimates;
  values above 64 are impractical at 2500 samples per segment on most
  research GPUs; powers of two maximise GPU memory throughput.

### Fixed architectural choices (not tuned)

**Layer normalisation (not batch normalisation):**
EEG amplitude varies across mice even after preprocessing. Batch normalisation
computes statistics across the batch dimension, making it statistically unreliable
at small batch sizes (16--64) and sensitive to between-subject amplitude differences.
Layer normalisation computes statistics per sample across the channel dimension,
so it is unaffected by batch composition.

**Spatial dropout (nn.Dropout1d):**
Standard dropout randomly zeroes individual scalar values. In a 1-D convolutional
network, adjacent time steps within a feature map carry correlated information.
Spatial dropout drops entire feature-map channels at once, which is a stronger
form of regularisation better suited to structured EEG representations.

**Weighted cross-entropy loss (BCEWithLogitsLoss with pos_weight):**
Ictal segments are rare relative to non-ictal. pos_weight = n_non_ictal / n_ictal
scales the loss on positive (ictal) samples so the model does not simply predict
the majority class. This is computed from training counts only.

**WeightedRandomSampler:**
Oversamples ictal segments during training so each batch is approximately balanced,
without discarding any data. This complements pos_weight: the sampler balances
what the model sees, while pos_weight adjusts how much each class contributes
to the gradient.

**Exponential dilation schedule (d_l = 2^l):**
The standard TCN default that maximises receptive field growth per additional layer.
Each layer doubles the dilation, so the RF grows exponentially with depth.

**AdamW with cosine annealing:**
AdamW decouples weight decay from the gradient update, preventing the regularisation
effect from shrinking as the learning rate decays. Cosine annealing gradually
reduces the learning rate from its initial value to near zero following a cosine
curve, avoiding abrupt drops that can destabilise training.

**GELU activation:**
A smooth approximation to ReLU that allows small negative gradients to flow.
Empirically well-suited to bio-signal feature extraction because EEG patterns
are smooth and continuous, not sharply thresholded.

**Global average pooling:**
After the final convolutional layer, global average pooling collapses the temporal
dimension (2500 time steps) to a single value per channel. This produces a
fixed-length vector regardless of minor variations in effective sequence length
after causal trimming, and acts as a strong spatial regulariser.

**Residual (skip) connections with 1x1 projection:**
Each convolutional block adds its input to its output. If the input and output
channel counts differ, a 1x1 convolution projects the input to match. This
stabilises gradient flow in deep stacks and prevents the network from losing
information as depth increases.

### How Optuna TPE works

**What is a trial?**
Each trial is one complete training run with a specific set of hyperparameters.
Optuna selects the hyperparameters, trains the model, evaluates it, and records
the result.

**What does TPE do differently from random search?**
Random search picks hyperparameters uniformly at random. TPE (Tree-structured
Parzen Estimator) builds two probability models after the first 15 trials:
one for hyperparameter configurations that produced good results, and one for
configurations that produced poor results. It then samples new configurations
that are likely under the "good" model and unlikely under the "bad" model.
This concentrates the search on promising regions of the hyperparameter space.

**What does the pruner do?**
The MedianPruner monitors validation F1 at each epoch. If a trial is performing
below the median of all previous trials at the same epoch, it is stopped early.
This saves compute by abandoning clearly poor configurations without waiting
for all 100 epochs.

**Early stopping (patience = 10):**
Within each trial, if the validation macro F1 does not improve for 10 consecutive
epochs, training stops and the best weights from the best epoch are restored.
This prevents overfitting and wasted computation on trials that have converged.

### GPU acceleration in this context

Optuna's search logic (TPE sampler, pruner, trial bookkeeping) runs entirely
on **CPU**. However, the computationally expensive operations inside each trial
-- the forward pass, loss computation, backward pass, and weight update --
all execute on the **GPU** when one is available. The result is that each trial
completes significantly faster, allowing more configurations to be evaluated
within a fixed time budget. Data is transferred to the GPU batch-by-batch
(not all at once) to avoid exhausting GPU memory on large datasets.

## Section 2 -- Install Dependencies

Run this cell only if packages are not already installed.

In [ ]:
# -- Section 2: Install dependencies (skip if already installed) ---------------
# !pip install torch optuna numpy scikit-learn matplotlib seaborn  # uncomment if needed

## Section 3 -- Imports, Reproducibility, and Device Configuration

In [ ]:
# -*- coding: utf-8 -*-
# -- Section 3: Imports, reproducibility, device detection ---------------------

import json                          # save hyperparameters and summary as JSON
import csv                           # write study results to CSV
import time                          # measure trial duration
import random                        # seed Python's built-in RNG
import logging                       # structured logging to file and console
from pathlib import Path             # cross-platform file path handling
from datetime import datetime        # ISO 8601 timestamp for summary

import numpy as np                   # numerical operations on arrays
import torch                         # deep learning framework
import torch.nn as nn                # neural network modules
import torch.nn.functional as F      # functional API (GELU, etc.)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # data pipeline
from torch.optim import AdamW        # optimiser with decoupled weight decay
from torch.optim.lr_scheduler import CosineAnnealingLR  # learning rate scheduler

from sklearn.metrics import f1_score  # macro F1 metric for evaluation

import optuna                        # hyperparameter optimisation framework
from optuna.samplers import TPESampler  # Tree-structured Parzen Estimator
from optuna.pruners import MedianPruner  # prune underperforming trials

import matplotlib                    # plotting backend configuration
matplotlib.use('Agg')               # non-interactive backend for cluster use
import matplotlib.pyplot as plt      # plotting API

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress Optuna's verbose output


def set_seed(seed=42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)                    # Python built-in RNG
    np.random.seed(seed)                 # NumPy RNG
    torch.manual_seed(seed)              # PyTorch CPU RNG
    torch.cuda.manual_seed_all(seed)     # PyTorch GPU RNG (all devices)
    torch.backends.cudnn.deterministic = True   # deterministic CUDA ops
    torch.backends.cudnn.benchmark = False      # disable auto-tuner for reproducibility


set_seed(42)  # set global seed immediately

# -- Device detection ----------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # select GPU if available

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)              # GPU model name
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9  # total VRAM in GB
    cuda_ver = torch.version.cuda                          # CUDA toolkit version
    print(f"Device        : {gpu_name}")
    print(f"VRAM total    : {vram_gb:.2f} GB")
    print(f"CUDA version  : {cuda_ver}")
else:
    gpu_name = "cpu"
    print("No GPU detected -- training will run on CPU.")
    print("Tuning 60 trials on CPU may take considerably longer.")

print(f"PyTorch       : {torch.__version__}")
print(f"Optuna        : {optuna.__version__}")
print(f"Using device  : {DEVICE}")

## Section 4 -- Configuration

All named constants are defined here. Paths use `pathlib.Path` throughout.
The test partition is intentionally excluded -- it is reserved for a separate
evaluation script and must not be referenced anywhere in this notebook.

In [ ]:
# -- Section 4: Configuration --------------------------------------------------

# -- Data paths ----------------------------------------------------------------
DATA_ROOT          = Path("data")                          # root directory for all partitions
TRAIN_ICTAL_DIR    = DATA_ROOT / "train" / "seizure"       # training ictal segments
TRAIN_NON_ICTAL_DIR = DATA_ROOT / "train" / "non_seizure"  # training non-ictal segments
VAL_ICTAL_DIR      = DATA_ROOT / "val" / "seizure"         # validation ictal segments
VAL_NON_ICTAL_DIR  = DATA_ROOT / "val" / "non_seizure"     # validation non-ictal segments

# -- Signal parameters ---------------------------------------------------------
FS            = 500    # sampling rate in Hz
SEGMENT_LEN   = 2500   # samples per segment: 5 s * 500 Hz

# -- Training protocol ---------------------------------------------------------
MAX_EPOCHS    = 100    # maximum epochs per trial before early stopping
ES_PATIENCE   = 10     # early stopping patience: epochs without val F1 improvement
GRAD_CLIP     = 1.0    # maximum gradient norm for gradient clipping
SEED          = 42     # random seed for reproducibility

# -- Optuna configuration ------------------------------------------------------
N_TRIALS      = 60     # total number of Optuna trials
N_STARTUP     = 15     # random exploration trials before TPE kicks in
STUDY_NAME    = "tcn_binary_seizure_hpt"  # Optuna study name

# -- Output --------------------------------------------------------------------
OUTPUT_DIR    = Path("outputs")       # directory for all saved outputs
OUTPUT_DIR.mkdir(exist_ok=True)       # create if it does not exist

# -- Logging setup -------------------------------------------------------------
log = logging.getLogger("tcn_hpt")    # named logger for this notebook
log.setLevel(logging.INFO)            # minimum log level
log.handlers.clear()                  # clear handlers from previous runs

fh = logging.FileHandler(OUTPUT_DIR / "tuning.log", mode="a", encoding="utf-8")  # file handler
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))  # timestamp format

ch = logging.StreamHandler()          # console handler
ch.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))  # same format

log.addHandler(fh)                    # attach file handler
log.addHandler(ch)                    # attach console handler

log.info(f"Configuration loaded. Device: {DEVICE}")
log.info(f"Output directory: {OUTPUT_DIR.resolve()}")

## Section 5 -- Dataset and DataLoader

Each `.npy` file is loaded on demand (not all at once) to keep memory usage low.
Per-segment z-score normalisation is applied when each segment is loaded:

```
z = (x - mean(x)) / (std(x) + 1e-8)
```

This input-level normalisation standardises the raw amplitude of each segment.
It does **not** replace LayerNorm inside the network, which corrects for
activation drift between convolutional layers during the forward pass.

`pin_memory=True` is set when a GPU is available. This locks DataLoader
worker memory pages so the CUDA DMA engine can transfer batches to GPU
without an intermediate copy, reducing per-batch transfer latency.

In [ ]:
# -- Section 5: Dataset and DataLoader -----------------------------------------

class EEGSegmentDataset(Dataset):
    """Memory-efficient EEG segment dataset that loads .npy files on demand."""

    def __init__(self, file_label_pairs):
        """
        Parameters
        ----------
        file_label_pairs : list of (Path, int)
            Each entry is (path_to_npy, label) where label is 0 or 1.
        """
        self.pairs = file_label_pairs  # store the list of (path, label) pairs

    def __len__(self):
        return len(self.pairs)  # total number of segments

    def __getitem__(self, idx):
        path, label = self.pairs[idx]              # get the file path and label for this index
        x = np.load(path).astype(np.float32)       # load the segment as float32
        mu = x.mean()                              # per-segment mean for z-score
        sigma = x.std() + 1e-8                     # per-segment std with epsilon to prevent div-by-zero
        x = (x - mu) / sigma                       # z-score normalise this segment
        x = torch.from_numpy(x).unsqueeze(0)       # shape: (1, SEGMENT_LEN) -- 1 channel
        y = torch.tensor(label, dtype=torch.float32)  # scalar label: 0.0 or 1.0
        return x, y


def collect_files(ictal_dir, non_ictal_dir):
    """Collect all .npy file paths with labels from two class directories.

    Returns
    -------
    pairs : list of (Path, int)
        Sorted list of (file_path, label) pairs. Label 1 = ictal, 0 = non-ictal.
    """
    pairs = []                                                # accumulate (path, label) pairs
    for p in sorted(Path(ictal_dir).glob("*.npy")):           # all ictal .npy files
        pairs.append((p, 1))                                  # label 1 = ictal (seizure)
    for p in sorted(Path(non_ictal_dir).glob("*.npy")):       # all non-ictal .npy files
        pairs.append((p, 0))                                  # label 0 = non-ictal
    return pairs


def make_loader(file_label_pairs, batch_size, train=True):
    """Build a DataLoader with optional WeightedRandomSampler for training.

    Parameters
    ----------
    file_label_pairs : list of (Path, int)
    batch_size : int
    train : bool
        If True, use WeightedRandomSampler to oversample the minority class.

    Returns
    -------
    loader : DataLoader
    """
    dataset = EEGSegmentDataset(file_label_pairs)  # create the dataset

    pin = (DEVICE.type == "cuda")  # pin memory for faster CPU-to-GPU transfer via DMA

    if train:
        labels = [lbl for _, lbl in file_label_pairs]  # extract all labels
        n_pos = sum(labels)                            # count ictal segments
        n_neg = len(labels) - n_pos                    # count non-ictal segments
        weight_per_class = {0: 1.0 / max(n_neg, 1),   # inverse frequency weight for non-ictal
                            1: 1.0 / max(n_pos, 1)}    # inverse frequency weight for ictal
        sample_weights = [weight_per_class[l] for l in labels]  # per-sample weight
        sampler = WeightedRandomSampler(               # oversample minority class
            weights=sample_weights,                    # sampling probability per segment
            num_samples=len(labels),                   # draw this many samples per epoch
            replacement=True                           # allow repeated draws for minority class
        )
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=sampler,         # balanced sampling replaces shuffle
            num_workers=0,           # 0 for cross-platform compatibility; increase on Linux
            pin_memory=pin,          # lock pages for faster GPU transfer when CUDA is available
            drop_last=False          # keep all segments including the last incomplete batch
        )
    else:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,           # no shuffling for validation -- deterministic order
            num_workers=0,           # 0 for cross-platform compatibility
            pin_memory=pin,          # lock pages for faster GPU transfer when CUDA is available
            drop_last=False
        )

    return loader


# -- Collect files and compute class statistics --------------------------------
train_pairs = collect_files(TRAIN_ICTAL_DIR, TRAIN_NON_ICTAL_DIR)  # training file-label pairs
val_pairs   = collect_files(VAL_ICTAL_DIR, VAL_NON_ICTAL_DIR)      # validation file-label pairs

n_train_ictal     = sum(1 for _, l in train_pairs if l == 1)  # training ictal count
n_train_non_ictal = sum(1 for _, l in train_pairs if l == 0)  # training non-ictal count
n_val_ictal       = sum(1 for _, l in val_pairs if l == 1)    # validation ictal count
n_val_non_ictal   = sum(1 for _, l in val_pairs if l == 0)    # validation non-ictal count

# pos_weight for BCEWithLogitsLoss: ratio of non-ictal to ictal in training set
POS_WEIGHT_VAL = n_train_non_ictal / max(n_train_ictal, 1)    # avoid division by zero

log.info(f"Train: {n_train_ictal} ictal + {n_train_non_ictal} non-ictal "
         f"= {len(train_pairs)} total ({100*n_train_ictal/max(len(train_pairs),1):.1f}% ictal)")
log.info(f"Val:   {n_val_ictal} ictal + {n_val_non_ictal} non-ictal "
         f"= {len(val_pairs)} total ({100*n_val_ictal/max(len(val_pairs),1):.1f}% ictal)")
log.info(f"pos_weight = {POS_WEIGHT_VAL:.4f}")

## Section 6 -- TCN Architecture

The model is moved to DEVICE immediately after instantiation with `.to(DEVICE)`.
This single call transfers all submodules, parameters, and buffers (including
LayerNorm statistics) to the target device.

In [ ]:
# -- Section 6: TCN Architecture -----------------------------------------------

class CausalConvBlock(nn.Module):
    """Single causal convolutional block with residual connection."""

    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        """
        Parameters
        ----------
        in_ch : int       Input channels.
        out_ch : int      Output channels.
        kernel_size : int Convolution kernel size (odd).
        dilation : int    Dilation factor for this layer.
        dropout : float   Spatial dropout rate.
        """
        super().__init__()
        self.pad = (kernel_size - 1) * dilation  # causal padding: left-pad only, trim right after conv
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              dilation=dilation, padding=self.pad)  # dilated causal conv
        self.norm = nn.LayerNorm(out_ch)   # normalise across channel dimension per time step
        self.act = nn.GELU()               # smooth activation for bio-signal features
        self.drop = nn.Dropout1d(dropout)  # spatial dropout: drops entire channels

        # Residual projection: 1x1 conv when channel counts differ
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        """Forward pass. x shape: (batch, channels, time)."""
        res = self.residual(x)                 # project input for residual addition
        out = self.conv(x)                     # dilated convolution with causal padding
        out = out[:, :, :x.size(2)]            # trim right to enforce causal (no future leak)
        out = out.transpose(1, 2)              # (B, T, C) for LayerNorm
        out = self.norm(out)                   # normalise across channel dimension
        out = out.transpose(1, 2)              # (B, C, T) back to conv format
        out = self.act(out)                    # GELU activation
        out = self.drop(out)                   # spatial dropout
        return out + res                       # residual connection


class TCN(nn.Module):
    """Temporal Convolutional Network for binary EEG classification."""

    def __init__(self, num_layers, num_filters, kernel_size, dropout):
        """
        Parameters
        ----------
        num_layers : int   Number of causal conv blocks.
        num_filters : int  Channels per convolutional layer.
        kernel_size : int  Kernel size (odd).
        dropout : float    Spatial dropout rate.
        """
        super().__init__()
        layers = []                             # accumulate blocks
        for i in range(num_layers):
            in_ch = 1 if i == 0 else num_filters  # first block takes 1-channel input
            dilation = 2 ** i                      # exponential dilation schedule
            layers.append(CausalConvBlock(in_ch, num_filters, kernel_size, dilation, dropout))
        self.network = nn.Sequential(*layers)   # stack all blocks sequentially
        self.head = nn.Linear(num_filters, 1)   # classification head: 1 logit for binary

        # Compute and store receptive field
        self.rf = (2 ** num_layers) * (kernel_size - 1)  # RF formula for exponential dilation
        log.info(f"TCN built: {num_layers} layers, {num_filters} filters, k={kernel_size}, "
                 f"RF={self.rf} samples ({self.rf/FS:.2f} s)")

    def forward(self, x):
        """Forward pass. x shape: (batch, 1, SEGMENT_LEN). Returns logits (batch,)."""
        out = self.network(x)                   # (batch, num_filters, time)
        out = out.mean(dim=2)                   # global average pooling over time
        return self.head(out).squeeze(-1)       # (batch,) raw logits


def count_parameters(model):
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Section 7 -- Training and Evaluation Utilities

Input and label tensors are moved to the device **batch-by-batch** inside
the training and evaluation loops. This avoids loading the entire dataset
into GPU memory at once, which would exhaust VRAM for large datasets.

Best model weights are stored on CPU (`best_state`) to avoid occupying
a second copy of the model in GPU memory.

In [ ]:
# -- Section 7: Training and Evaluation Utilities ------------------------------

def train_one_epoch(model, loader, optimiser, criterion, device):
    """Train the model for one epoch.

    Returns
    -------
    mean_loss : float
        Average training loss over all batches.
    """
    model.train()                                  # set model to training mode
    total_loss = 0.0                               # accumulate batch losses
    n_batches = 0                                  # count batches

    for x, y in loader:
        x, y = x.to(device), y.to(device)         # transfer batch to device (batch-by-batch to save VRAM)
        optimiser.zero_grad()                      # clear gradients from previous step
        logits = model(x)                          # forward pass on GPU
        loss = criterion(logits, y)                # compute weighted BCE loss
        loss.backward()                            # backward pass: compute gradients
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)  # clip gradients to prevent explosion
        optimiser.step()                           # update weights
        total_loss += loss.item()                  # accumulate scalar loss (detached from graph)
        n_batches += 1

    return total_loss / max(n_batches, 1)          # return mean loss


@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model on a dataset and return macro F1.

    Returns
    -------
    macro_f1 : float
    y_true : np.ndarray
    y_pred : np.ndarray
    """
    model.eval()                                   # set model to evaluation mode
    all_true = []                                  # accumulate true labels on CPU
    all_pred = []                                  # accumulate predicted labels on CPU

    for x, y in loader:
        x = x.to(device)                          # transfer input to device
        logits = model(x)                          # forward pass
        preds = (torch.sigmoid(logits) >= 0.5).long()  # threshold at 0.5 for binary prediction
        all_true.append(y.cpu().numpy())           # move to CPU to avoid VRAM accumulation
        all_pred.append(preds.cpu().numpy())       # move to CPU to avoid VRAM accumulation

    y_true = np.concatenate(all_true)              # flatten into 1-D array
    y_pred = np.concatenate(all_pred)              # flatten into 1-D array
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)  # macro F1

    return macro_f1, y_true, y_pred


def run_training(model, train_loader, val_loader, lr, weight_decay,
                 max_epochs, patience, trial, device):
    """Full training loop with early stopping and Optuna pruning.

    Returns
    -------
    best_val_f1 : float
        Best validation macro F1 achieved during training.
    """
    # pos_weight must be on the same device as model output
    pw = torch.tensor([POS_WEIGHT_VAL], dtype=torch.float32).to(device)  # move to device for loss
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)  # weighted binary cross-entropy

    optimiser = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)  # decoupled weight decay
    scheduler = CosineAnnealingLR(optimiser, T_max=max_epochs,
                                  eta_min=lr * 0.01)  # cosine annealing to near-zero lr

    best_val_f1 = 0.0                              # track best validation F1
    epochs_no_improve = 0                          # early stopping counter
    best_state = None                              # store best weights on CPU to save VRAM

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, device)
        val_f1, _, _ = evaluate(model, val_loader, device)
        scheduler.step()                           # advance cosine annealing schedule

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1                   # update best F1
            epochs_no_improve = 0                  # reset patience counter
            # Store best weights on CPU to avoid a second GPU copy
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1                 # increment patience counter

        # Optuna pruning check
        trial.report(val_f1, epoch)                # report intermediate F1 to Optuna
        if trial.should_prune():                   # check if this trial should be pruned
            # Restore best weights before pruning (for consistent reporting)
            if best_state is not None:
                model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
            raise optuna.exceptions.TrialPruned()

        if epochs_no_improve >= patience:          # early stopping triggered
            break

    # Restore best weights at end of training
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return best_val_f1

## Section 8 -- Optuna Objective Function

Each call to `optuna_objective(trial)` is one complete experiment:
Optuna proposes hyperparameters, the model is built, trained, evaluated,
and the best validation F1 is returned. Optuna uses this value to guide
the next proposal via TPE.

In [ ]:
# -- Section 8: Optuna Objective Function --------------------------------------

def optuna_objective(trial):
    """Optuna objective: train a TCN with proposed hyperparameters, return val F1.

    Parameters
    ----------
    trial : optuna.trial.Trial

    Returns
    -------
    best_val_f1 : float
    """
    # -- Sample hyperparameters ------------------------------------------------
    num_layers = trial.suggest_int("num_layers", 5, 9)                 # depth of the TCN
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])  # odd kernel sizes only
    num_filters = trial.suggest_categorical("num_filters", [32, 64, 128])  # channel width
    dropout = trial.suggest_float("dropout", 0.10, 0.50, step=0.05)   # spatial dropout rate
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)   # AdamW learning rate
    wd = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)    # L2 regularisation
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64]) # segments per batch

    # -- Check receptive field constraint --------------------------------------
    rf = (2 ** num_layers) * (kernel_size - 1)  # compute RF for this configuration
    if rf < 500:                                 # must cover at least 1 second at 500 Hz
        raise optuna.exceptions.TrialPruned()    # reject this configuration immediately

    # -- Build model -----------------------------------------------------------
    set_seed(SEED)                               # ensure reproducible weight initialisation
    model = TCN(num_layers, num_filters, kernel_size, dropout).to(DEVICE)  # move model to GPU/CPU

    log.info(f"Trial {trial.number}: L={num_layers} k={kernel_size} f={num_filters} "
             f"drop={dropout:.2f} lr={lr:.2e} wd={wd:.2e} bs={batch_size} "
             f"RF={rf} params={count_parameters(model)}")

    # -- Build data loaders ----------------------------------------------------
    train_loader = make_loader(train_pairs, batch_size, train=True)   # balanced training loader
    val_loader   = make_loader(val_pairs, batch_size, train=False)    # deterministic val loader

    # -- Train and evaluate ----------------------------------------------------
    best_val_f1 = run_training(
        model, train_loader, val_loader,
        lr=lr, weight_decay=wd,
        max_epochs=MAX_EPOCHS, patience=ES_PATIENCE,
        trial=trial, device=DEVICE
    )

    # -- Save checkpoint for this trial ----------------------------------------
    ckpt_path = OUTPUT_DIR / f"trial_{trial.number:03d}.pt"
    torch.save(model.cpu().state_dict(), ckpt_path)  # save on CPU for device-agnostic loading

    return best_val_f1

## Section 9 -- Run the Hyperparameter Search

The study is created with TPESampler and MedianPruner. A callback logs
each completed trial. After all trials, results are printed and GPU
memory is released.

In [ ]:
# -- Section 9: Run the Hyperparameter Search ----------------------------------

def trial_callback(study, trial):
    """Callback executed after each completed trial."""
    if trial.state == optuna.trial.TrialState.COMPLETE:
        p = trial.params
        log.info(f"  [Trial {trial.number:3d}] F1={trial.value:.4f} "
                 f"L={p['num_layers']} k={p['kernel_size']} f={p['num_filters']} "
                 f"lr={p['learning_rate']:.2e} device={DEVICE.type}")


# -- Create study --------------------------------------------------------------
sampler = TPESampler(seed=SEED, n_startup_trials=N_STARTUP)  # TPE with 15 random starts
pruner  = MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=15)  # prune below median

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",        # maximise validation macro F1
    sampler=sampler,
    pruner=pruner
)

log.info(f"Starting Optuna study: {N_TRIALS} trials, TPE sampler, MedianPruner")
log.info(f"Training device: {DEVICE.type}")

study.optimize(
    optuna_objective,
    n_trials=N_TRIALS,
    callbacks=[trial_callback],
    show_progress_bar=False      # disabled for cluster/log compatibility
)

# -- Print results -------------------------------------------------------------
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

log.info(f"\n{'='*60}")
log.info(f"Study complete: {len(completed)} completed, {len(pruned)} pruned "
         f"out of {len(study.trials)} total")

best = study.best_trial
bp = best.params
best_rf = (2 ** bp["num_layers"]) * (bp["kernel_size"] - 1)
rf_check = "PASS" if best_rf >= 500 else "WARNING: RF < 500"

log.info(f"Best trial     : {best.number}")
log.info(f"Best val F1    : {best.value:.6f}")
log.info(f"  num_layers   : {bp['num_layers']}")
log.info(f"  kernel_size  : {bp['kernel_size']}")
log.info(f"  num_filters  : {bp['num_filters']}")
log.info(f"  dropout      : {bp['dropout']:.2f}")
log.info(f"  learning_rate: {bp['learning_rate']:.2e}")
log.info(f"  weight_decay : {bp['weight_decay']:.2e}")
log.info(f"  batch_size   : {bp['batch_size']}")
log.info(f"  RF           : {best_rf} samples ({best_rf/FS:.2f} s) [{rf_check}]")
log.info(f"  Device       : {DEVICE.type}")

# -- Release GPU memory --------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()     # free cached GPU memory
    log.info("GPU cache cleared.")

## Section 10 -- Visualise Tuning Results

All plots are saved to the outputs/ directory as PNG files.
The matplotlib Agg backend is used so plots render without a display server.

In [ ]:
# -- Section 10: Visualise Tuning Results --------------------------------------

completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed_trials]    # trial indices
trial_f1s  = [t.value for t in completed_trials]     # corresponding val F1 values

# -- Plot 1: F1 history -------------------------------------------------------
running_best = np.maximum.accumulate(trial_f1s)      # running best F1 up to each trial
best_idx = int(np.argmax(trial_f1s))                 # index of the best trial in the list

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(trial_nums, trial_f1s, alpha=0.6, s=30, label="Trial F1")  # scatter of all F1 values
ax.plot(trial_nums, running_best, color="red", linewidth=2, label="Running best")  # best-so-far line
ax.scatter([trial_nums[best_idx]], [trial_f1s[best_idx]],
           color="gold", s=150, zorder=5, edgecolors="black", marker="*",
           label=f"Best: {trial_f1s[best_idx]:.4f}")  # highlight best trial
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation macro F1")
ax.set_title("Hyperparameter Tuning -- F1 History")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "tuning_f1_history.png", dpi=150)  # save before showing
plt.close(fig)
log.info(f"Saved: {OUTPUT_DIR / 'tuning_f1_history.png'}")

# -- Plot 2: F1 distribution --------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(trial_f1s, bins=20, edgecolor="black", alpha=0.7)  # histogram of F1 values
best_f1 = max(trial_f1s)
ax.axvline(best_f1, color="red", linestyle="--", linewidth=2,
           label=f"Best: {best_f1:.4f}")  # vertical line at best F1
ax.set_xlabel("Validation macro F1")
ax.set_ylabel("Count")
ax.set_title("Hyperparameter Tuning -- F1 Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "tuning_f1_distribution.png", dpi=150)
plt.close(fig)
log.info(f"Saved: {OUTPUT_DIR / 'tuning_f1_distribution.png'}")

# -- Plot 3: Hyperparameter importance ----------------------------------------
try:
    importances = optuna.importance.get_param_importances(study)  # compute importance scores
    params_sorted = list(importances.keys())      # parameter names sorted by importance
    values_sorted = list(importances.values())    # corresponding importance values

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["tomato" if i == 0 else "steelblue" for i in range(len(params_sorted))]  # highlight top
    ax.barh(params_sorted[::-1], values_sorted[::-1], color=colors[::-1])  # horizontal bars
    ax.set_xlabel("Importance")
    ax.set_title("Hyperparameter Importance")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "hyperparameter_importance.png", dpi=150)
    plt.close(fig)
    log.info(f"Saved: {OUTPUT_DIR / 'hyperparameter_importance.png'}")
except Exception as e:
    log.warning(f"Could not compute hyperparameter importance: {e}")

# -- Plot 4: Parallel coordinate plot -----------------------------------------
try:
    from optuna.visualization.matplotlib import plot_parallel_coordinate
    fig = plot_parallel_coordinate(study)          # Optuna's built-in parallel coordinate plot
    fig.figure.tight_layout()
    fig.figure.savefig(OUTPUT_DIR / "parallel_coordinates.png", dpi=150)
    plt.close(fig.figure)
    log.info(f"Saved: {OUTPUT_DIR / 'parallel_coordinates.png'}")
except Exception as e:
    log.warning(f"Could not create parallel coordinate plot: {e}")

## Section 11 -- Save All Tuning Outputs

Three files are saved:

- **best_params.json** -- The winning hyperparameters. Used by the retraining
  script to rebuild the best model architecture and train on the full
  train+val dataset before final test evaluation.
- **study_results.csv** -- One row per completed trial. Useful for post-hoc
  analysis of which configurations worked and which did not.
- **tuning_summary.json** -- Metadata about the study: timestamps, trial
  counts, device info, and signal parameters. Provides a complete audit
  trail for reproducibility.

In [ ]:
# -- Section 11a: Save best hyperparameters ------------------------------------

bp = study.best_trial.params
best_rf = (2 ** bp["num_layers"]) * (bp["kernel_size"] - 1)

best_params = {
    "best_trial_number": study.best_trial.number,
    "best_val_f1": round(study.best_trial.value, 6),
    "receptive_field_samples": best_rf,
    "receptive_field_seconds": round(best_rf / FS, 4),
    "training_device": DEVICE.type,
    "hyperparameters": {
        "num_layers":    bp["num_layers"],
        "kernel_size":   bp["kernel_size"],
        "num_filters":   bp["num_filters"],
        "dropout":       bp["dropout"],
        "learning_rate": bp["learning_rate"],
        "weight_decay":  bp["weight_decay"],
        "batch_size":    bp["batch_size"]
    }
}

params_path = OUTPUT_DIR / "best_params.json"
with open(params_path, "w") as f:
    json.dump(best_params, f, indent=2)
log.info(f"Saved: {params_path.resolve()}")

In [ ]:
# -- Section 11b: Save full study results as CSV -------------------------------

csv_path = OUTPUT_DIR / "study_results.csv"
fieldnames = ["trial_number", "val_f1", "num_layers", "kernel_size", "num_filters",
              "dropout", "learning_rate", "weight_decay", "batch_size",
              "duration_seconds", "device"]

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for t in completed:
        dur = (t.datetime_complete - t.datetime_start).total_seconds() if t.datetime_complete else 0
        writer.writerow({
            "trial_number":    t.number,
            "val_f1":          round(t.value, 6),
            "num_layers":      t.params["num_layers"],
            "kernel_size":     t.params["kernel_size"],
            "num_filters":     t.params["num_filters"],
            "dropout":         t.params["dropout"],
            "learning_rate":   t.params["learning_rate"],
            "weight_decay":    t.params["weight_decay"],
            "batch_size":      t.params["batch_size"],
            "duration_seconds": round(dur, 1),
            "device":          DEVICE.type
        })

log.info(f"Saved: {csv_path.resolve()}")

In [ ]:
# -- Section 11c: Save tuning summary -----------------------------------------

summary = {
    "timestamp":              datetime.now().isoformat(),
    "study_name":             STUDY_NAME,
    "n_trials_requested":     N_TRIALS,
    "n_trials_completed":     len([t for t in study.trials
                                   if t.state == optuna.trial.TrialState.COMPLETE]),
    "n_trials_pruned":        len([t for t in study.trials
                                   if t.state == optuna.trial.TrialState.PRUNED]),
    "best_trial_number":      study.best_trial.number,
    "best_val_f1":            round(study.best_trial.value, 6),
    "training_device":        DEVICE.type,
    "gpu_name":               gpu_name,
    "fs_hz":                  FS,
    "segment_len_samples":    SEGMENT_LEN,
    "segment_len_seconds":    SEGMENT_LEN / FS
}

summary_path = OUTPUT_DIR / "tuning_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
log.info(f"Saved: {summary_path.resolve()}")

# -- Confirm all outputs -------------------------------------------------------
log.info("")
log.info("All tuning outputs saved successfully:")
log.info(f"  1. {params_path.resolve()}")
log.info(f"  2. {csv_path.resolve()}")
log.info(f"  3. {summary_path.resolve()}")
log.info("Tuning notebook complete.")

## End of notebook

This notebook is now complete. The best hyperparameters have been saved to
`outputs/best_params.json`. The next step is to use a separate retraining
script that loads these parameters, rebuilds the TCN with the winning
configuration, trains on the combined train+val partition, and evaluates
on the held-out test set.

In [ ]:
# -- End of notebook -----------------------------------------------------------
log.info("Notebook execution finished.")